<a href="https://colab.research.google.com/github/faresayman-ai/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# Scoring.
# because each page/query pair gets an independent, absolute score — the gap between expected CTR without the need to look for informations from other row

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
# gap score, the difference between the expected CTR - expected by regression ML model - and the actual CTR from the table.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
# score = expected_ctr(position, intent, device) - actual_ctr
# if score = 0 means the page is performing as it should be
# if score is large and positive means the page underperforming
# if score is negative means the page is performing more than expected

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
le = LabelEncoder()

df['trend_direction_enc'] = le.fit_transform(df['trend_direction'])
df['impression_tier_enc'] = le.fit_transform(df['impression_tier'])

for col in ['age_tier','freshness_tier']:
  processed_col = df[col].replace(r'(\d+)\+', r'\1-\1', regex=True)
  df[['min_val', 'max_val']] = processed_col.str.split('-', expand=True).astype(float)
  df['mid_val'] = (df['min_val'] + df['max_val']) / 2
X = df[['trend_direction_enc', 'impression_tier_enc', 'mid_val']]
y = df['ctr']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = XGBRegressor(max_depth=3)
model.fit(X_train, y_train)
expected_ctr = model.predict(X)
score = expected_ctr - df['ctr']

print(pd.DataFrame({'content_id': df['content_id'], 'score': score}))

                 content_id     score
0      content_304f48230142 -0.490828
1      content_a1fb4e703a9e  0.219172
2      content_9aa793d4d895  0.179172
3      content_331d6c4de07b -0.098526
4      content_d99b7a2d90ca  0.139172
...                     ...       ...
29995  content_c322796023c8  1.377684
29996  content_526572edb3fa -0.173486
29997  content_38112bdd0c6e  0.079172
29998  content_ab26273a7e7a  0.074322
29999  content_887020f20b5e -1.517606

[30000 rows x 2 columns]


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
# Expected CTR depends on several factors interacting at once — position/impression tier,
# trend direction, and content age/freshness don't move the needle independently.
# A fixed rule (e.g. "flag anything below 2% CTR") ignores that a low-impression,
# declining-trend, old page is EXPECTED to have low CTR, while the same CTR on a
# high-impression, trending-up, fresh page is a real problem.
# A hardcoded rule would need a separate threshold for every combination of tier x
# trend x age — that's dozens of if-statements, brittle, and still wouldn't capture
# nonlinear interactions (e.g. freshness mattering more when trend is rising than when it's flat).
# The regression model learns expected_ctr(position, intent, device) as a smooth function
# from data, so the "expected" baseline adapts automatically instead of being manually tuned.

## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.